# 24. M7 검색 개선 전후 결과 읽기

[프로젝트 적용] 같은 PDF·청크·모델·질문 벡터·30문항 기대 출처로 비교했습니다.
[프로젝트 추가] 한 가지 변경은 **원문에 실제로 들어 있는 구체적인 질문 표현의 점수**입니다. 특정 정답 페이지를 고정하지 않습니다.

이 노트북은 이미 계산한 결과 파일만 읽습니다. DB 조회·모델 실행·재임베딩을 하지 않습니다. 기존 검색 개선에 사용한 질문이며 독립된 새 질문 평가 또는 그림 이해·답변 정확도 평가가 아닙니다.


In [ ]:
from pathlib import Path
import json

# 실행 폴더가 notebooks여도 실제 프로젝트의 결과 파일을 찾습니다.
project = Path.cwd().resolve()
if project.name == "notebooks":
    project = project.parent
report_path = project / "data/zzong_santafe_lag/reports/m7_specific_terms_20261002_142212.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
if report["phase"] != "completed":
    raise ValueError("완료된 결과 파일을 먼저 확인하세요.")
print("평가 질문 수:", report["question_count"])
print("이전 사용자 결과와 기대 근거 순위 일치:", report["before_matches_user_reported_ranks"])


In [ ]:
# Hit@5는 찾은 비율이고 MRR@5는 앞쪽에서 찾을수록 높아지는 평균 점수입니다.
for label, key in (("변경 전", "before"), ("변경 후", "after")):
    metrics = report["metrics"][key]
    print(label, "| Hit@5:", round(metrics["Hit@5"], 3),
          "| MRR@5:", round(metrics["MRR@5"], 3),
          "| 1위에서 찾은 질문:", metrics["top1_count"])


In [ ]:
# 차단 클립 질문에서 어떤 원문 표현이 점수를 더 받았는지 확인합니다.
for label, key in (("변경 전", "before"), ("변경 후", "after")):
    result = next(row for row in report["results"][key] if row["case_id"] == "Q19")
    print("\n", label, "| 기대 근거 순위:", result["first_relevant_rank"])
    for hit in result["candidates"]:
        print(hit["rank"], hit["title"], "| PDF:", hit["pdf_pages"],
              "| 일치한 구체 표현:", hit["specific_terms"])


## 결과와 다음 작업

- 기존 30문항: Hit@5는 1.0 유지, MRR@5는 0.931 → 0.956, Q19 기대 근거는 4위 → 1위입니다. 나머지 29문항의 기대 근거 순위는 같습니다.
- Q16·Q17은 다른 유효한 근거가 앞에 나올 수 있습니다. 기존 기대 출처를 결과에 맞춰 바꾸지 않았습니다.
- 발췌 답변 서비스는 purpose_specific을 기본 사용합니다. 23번 노트북의 purpose는 변경 전 기준이며 두 평가를 섞지 않습니다.
- 같은 단어가 없는 바꿔 말한 질문·여러 조건 질문·근거 없는 질문을 추가로 확인해야 합니다. 모든 그림·자동 주제 경계 검토가 끝난 것은 아닙니다.

코드 역할: specific_terms.py는 추가 점수, retrieval.py는 순위, m7_search_experiment.py는 두 방식 비교·결과 기록입니다.
